# Contribution decay: CockroachDB to Polars

This notebook reproduces the existing `calculate_decay_scores` and `calculate_global_decay_scores` SQL functions. CockroachDB only performs the join, filter, and ordered streaming read; all decay decisions and score calculations happen in the local compute layer.

**Memory model:** rows are fetched with a server-side cursor in bounded batches. Only one batch, one output batch, and the current replier's 30-day rolling state are held in memory. `active_multipliers` is omitted by default because storing a list snapshot on every output row can grow quadratically.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import polars as pl

from mindshare_compute.db import iter_decay_source, read_golden_scores
from mindshare_compute.decay import DecayComputer, compare_scores

pl.Config.set_tbl_rows(12)
pl.show_versions()

--------Version info---------
Polars:              1.41.2
Index type:          UInt32
Platform:            Linux-6.18.33.1-microsoft-standard-WSL2-x86_64-with-glibc2.39
Python:              3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Runtime:             rt32

----Optional dependencies----
Azure CLI            <not installed>
adbc_driver_manager  <not installed>
altair               <not installed>
azure.identity       <not installed>
boto3                <not installed>
cloudpickle          <not installed>
connectorx           <not installed>
deltalake            <not installed>
fastexcel            <not installed>
fsspec               <not installed>
gevent               <not installed>
google.auth          <not installed>
great_tables         <not installed>
matplotlib           <not installed>
numpy                2.4.6
openpyxl             <not installed>
pandas               3.0.3
polars_cloud         <not installed>
pyarrow              24.0.0
pydantic             <not ins

## Configuration

Put the CockroachDB URI in `MINDSHARE_DB_URI` inside `.env`. For comparison against the existing PostgreSQL-generated scores, also set `MINDSHARE_PG_URI`.

Start with one project. `BATCH_SIZE` controls peak memory; lower it if WSL memory is tight. Writing each result batch to Parquet avoids retaining the full result in memory.

In [2]:
SCOPE = "project"           # "project" or "global"
PROJECT_KEYWORD = "quipnetwork" # ignored for global scope
BATCH_SIZE = 100_000
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = Path("output") / f"{SCOPE}_{PROJECT_KEYWORD}_{RUN_ID}"
WRITE_PARQUET = True
KEEP_RESULTS_IN_MEMORY = False  # only enable for a project known to fit in RAM

## Run and time the computation

`database_read_seconds` and `decay_compute_seconds` are measured separately. The decay timer covers the local stateful algorithm and creation of each Polars result batch. The wall-clock total also includes optional Parquet writes and notebook overhead.

In [3]:
if WRITE_PARQUET:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

computer = DecayComputer(SCOPE)
result_batches = []
rows_computed = 0
database_read_seconds = 0.0
decay_compute_seconds = 0.0
wall_start = perf_counter()

source = iter_decay_source(
    SCOPE,
    PROJECT_KEYWORD if SCOPE == "project" else None,
    target="crdb",
    batch_size=BATCH_SIZE,
)

batch_number = 0
while True:
    read_start = perf_counter()
    try:
        source_batch = next(source)
    except StopIteration:
        database_read_seconds += perf_counter() - read_start
        break
    database_read_seconds += perf_counter() - read_start

    compute_start = perf_counter()
    result_batch = computer.process_batch(source_batch)
    decay_compute_seconds += perf_counter() - compute_start

    rows_computed += result_batch.height
    if WRITE_PARQUET:
        result_batch.write_parquet(OUTPUT_DIR / f"part-{batch_number:05d}.parquet")
    if KEEP_RESULTS_IN_MEMORY:
        result_batches.append(result_batch)
    batch_number += 1

timing = pl.DataFrame({
    "rows_computed": [rows_computed],
    "database_read_seconds": [database_read_seconds],
    "decay_compute_seconds": [decay_compute_seconds],
    "total_wall_seconds": [perf_counter() - wall_start],
})
timing

rows_computed,database_read_seconds,decay_compute_seconds,total_wall_seconds
i64,f64,f64,f64
2216570,336.739648,19.182705,356.868685


## Inspect results

Scan the Parquet parts lazily so inspection does not load the entire output. The aggregation below is executed by Polars against the result files.

In [4]:
if WRITE_PARQUET:
    parquet_paths = list(OUTPUT_DIR.glob("*.parquet"))
    if not parquet_paths:
        raise FileNotFoundError(
            f"No parquet files found in {OUTPUT_DIR}. "
            "Write results to parquet before inspecting them."
        )
    results = pl.scan_parquet(OUTPUT_DIR / "*.parquet")
    display(results.head(10).collect())
    display(
        results.group_by("decay_type")
        .agg(pl.len().alias("rows"), pl.col("contribution_score").mean().alias("mean_score"))
        .sort("decay_type")
        .collect()
    )
elif KEEP_RESULTS_IN_MEMORY:
    results = pl.concat(result_batches)
    display(results.head(10))


project_keyword,reply_post_id,original_post_id,replier_x_id,original_author_x_id,post_created_at,replier_base_score,effective_score,contribution_score,reply_number,local_reply_count,decay_type
str,str,str,str,str,"datetime[μs, UTC]",f64,f64,f64,i64,i64,str
"""quipnetwork""","""2044210856461062512""","""2044202354291904575""","""""","""2285666115""",2026-04-15 00:27:02 UTC,0.0,0.0,0.0,1,1,"""FIRST_REPLY"""
"""quipnetwork""","""2044211242013712840""","""2043365730612301879""","""""","""1437682537149526019""",2026-04-15 00:28:34 UTC,0.0,0.0,0.0,2,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212093352849894""","""2044210505779425534""","""""","""""",2026-04-15 00:31:57 UTC,0.0,0.0,0.0,3,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212454591475833""","""2044212315600629869""","""""","""1719270086543040512""",2026-04-15 00:33:23 UTC,0.0,0.0,0.0,4,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212538334724557""","""2042783412885557584""","""""","""931391403350884353""",2026-04-15 00:33:43 UTC,0.0,0.0,0.0,5,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212581225607305""","""2042757777379201470""","""""","""1619588940775960576""",2026-04-15 00:33:53 UTC,0.0,0.0,0.0,6,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044219829708828956""","""2044219077045170329""","""""","""896279370""",2026-04-15 01:02:41 UTC,0.0,0.0,0.0,7,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044219829641720237""","""2044213824837038160""","""""","""1482396650471661572""",2026-04-15 01:02:41 UTC,0.0,0.0,0.0,8,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044220059959402861""","""2044219077045170329""","""""","""896279370""",2026-04-15 01:03:36 UTC,0.0,0.0,0.0,9,2,"""LOCAL_DECAY"""


decay_type,rows,mean_score
str,u32,f64
"""FIRST_REPLY""",39319,76.238711
"""GLOBAL_DECAY""",532784,14.530313
"""LOCAL_DECAY""",1644467,2.894047


In [5]:
# results.filter(replier_x_id='1001709042551738370').show()

## Compare with PostgreSQL SQL output

This validation reads the existing SQL-generated score table from PostgreSQL. It intentionally materializes both sides, so use it for one project rather than the global dataset. For a meaningful comparison, CockroachDB and PostgreSQL must contain the same source-data snapshot. A mismatch can also result from two replies by the same user having exactly the same timestamp: the original SQL has no tie-breaker in its `ORDER BY`.

In [ ]:
if SCOPE == "project" and WRITE_PARQUET:
    parquet_paths = list(OUTPUT_DIR.glob("*.parquet"))
    if not parquet_paths:
        raise FileNotFoundError(
            f"No parquet files found in {OUTPUT_DIR}. "
            "Run the cell that writes output parquet files first."
        )
    actual = pl.scan_parquet(OUTPUT_DIR / "*.parquet").collect()
    expected = read_golden_scores("project", PROJECT_KEYWORD, target="pg")
    mismatches = compare_scores(actual, expected)
    print(f"Compared {actual.height:,} rows; mismatches: {mismatches.height:,}")
    display(mismatches.head(20))
